# Check the pretrained ECGFounder encoder
Use the **symile-drfuse** kernel and run the cells in order. This notebook loads **one training ECG** from SYMILE-MIMIC, verifies its source metadata and lead order, applies ECGFounder's filtering and record-wide z-score normalization, and extracts a **1,024-value embedding** using the official pretrained weights. The final section repeats the checks on **20 additional training ECGs from distinct patients**.

For verification, it downloads only the selected public WFDB headers and waveforms (about 120 KB of waveform data per recording), then caches them inside the existing ignored dataset directory. It does not download the full MIMIC dataset, train a model, change the stored arrays, or modify Pipeline2's current encoders.

Sources: [ECGFounder repository](https://github.com/PKUDigitalHealth/ECGFounder), [official pretrained checkpoint](https://huggingface.co/PKUDigitalHealth/ECGFounder/blob/main/12_lead_ECGFounder.pth), and [MIMIC-IV-ECG documentation](https://physionet.org/content/mimic-iv-ecg/1.0/).

In [ ]:
from pathlib import Path
import hashlib
import importlib.util
import sys
from concurrent.futures import ThreadPoolExecutor

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import torch
from IPython.display import display
import wfdb

PROJECT_DIR = next(
    folder if (folder / "models/main_model.py").is_file() else folder / "Pipeline2"
    for folder in [Path.cwd(), *Path.cwd().parents]
    if (folder / "models/main_model.py").is_file()
    or (folder / "Pipeline2/models/main_model.py").is_file()
)
ECGFOUNDER_DIR = PROJECT_DIR.parent / "ecgfounder"
DATASET_ROOT = PROJECT_DIR.parent / "symile-mimic/symile-mimic-aws"
CHECKPOINT_PATH = ECGFOUNDER_DIR / "checkpoint/12_lead_ECGFounder.pth"
SAMPLE_INDEX = 0
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("ECGFounder code:", ECGFOUNDER_DIR)
print("Checkpoint:", CHECKPOINT_PATH)
print("Device:", DEVICE)

## Load one SYMILE training ECG
The array contains `(N, 1, 5000, 12)` values. The CSV is used only to identify the matching source record and verify admission alignment. SYMILE's saving script applies **one global min-max transform across the entire recording**, preserving the source channel order.

In [ ]:
split_dir = DATASET_ROOT / "data_npy/train"
ecg_array = np.load(split_dir / "ecg_train.npy", mmap_mode="r")
hadm_ids = np.load(split_dir / "hadm_id_train.npy", mmap_mode="r").reshape(-1)
metadata = pd.read_csv(DATASET_ROOT / "train.csv")
assert np.array_equal(hadm_ids, metadata["hadm_id"].to_numpy())

sample = np.array(ecg_array[SAMPLE_INDEX], dtype=np.float64, copy=True)
row = metadata.iloc[SAMPLE_INDEX]
assert sample.shape == (1, 5000, 12)
assert np.isfinite(sample).all()
print("Admission:", int(row["hadm_id"]))
print("Source ECG:", row["ecg_path"])
print("Stored shape:", sample.shape)
print("Stored min / max:", sample.min(), sample.max())

## Verify sampling rate and actual lead order
A shape of 5,000 samples alone does not prove the sampling rate or lead order. Read the **matching source header** to verify these. The source documentation specifies 10-second, 500 Hz ECGs. The source order in the current sample has `aVF` before `aVL`, while ECGFounder's `new_leads` list puts `aVL` before `aVF`.

The first run needs an internet connection for this one-record check. Later runs reuse the cached files. If downloading is unavailable, this cell reports the network error rather than silently assuming the metadata.

In [ ]:
def cached_source_record(ecg_path):
    relative_record = Path(str(ecg_path))
    if relative_record.is_absolute() or ".." in relative_record.parts:
        raise ValueError("Expected a relative MIMIC ECG record path.")
    record_name = relative_record.name
    cache_dir = DATASET_ROOT / "ecgfounder_sample_checks" / record_name
    cache_dir.mkdir(parents=True, exist_ok=True)
    base_url = "https://physionet.org/files/mimic-iv-ecg/1.0/"
    for extension in ["hea", "dat"]:
        destination = cache_dir / f"{record_name}.{extension}"
        if not destination.is_file() or destination.stat().st_size == 0:
            response = requests.get(
                f"{base_url}{relative_record.as_posix()}.{extension}", timeout=30,
            )
            response.raise_for_status()
            temporary = destination.with_suffix(destination.suffix + ".tmp")
            temporary.write_bytes(response.content)
            temporary.replace(destination)
    return cache_dir / record_name

local_record = cached_source_record(row["ecg_path"])
header = wfdb.rdheader(str(local_record))
assert header.fs == 500, f"Unexpected sampling rate: {header.fs}"
assert header.n_sig == 12 and header.sig_len == 5000
SOURCE_LEADS = list(header.sig_name)
FS = int(header.fs)
# Target order in ecgfounder/dataset.py; source order is verified from each header.
TARGET_LEADS = ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"]
lead_indices = [SOURCE_LEADS.index(name) for name in TARGET_LEADS]
print("Verified sampling rate:", FS, "Hz")
print("Verified duration:", header.sig_len / FS, "seconds")
print("Source lead order:", SOURCE_LEADS)
print("ECGFounder order:", TARGET_LEADS)
print("Reordering indices:", lead_indices)

## Check that the source record matches the stored sample
Reapply SYMILE's global min-max transformation to the raw source waveform and compare it with the selected NPY sample. This checks that the admission, record, time axis and channel positions are aligned before we use the header metadata.

In [ ]:
raw_record = wfdb.rdrecord(str(local_record))
raw_signal = raw_record.p_signal.astype(np.float64)  # (5000, 12), physical units
assert raw_signal.shape == (5000, 12) and np.isfinite(raw_signal).all()
span = raw_signal.max() - raw_signal.min()
assert span > 0
raw_minmax = 2 * (raw_signal - raw_signal.min()) / span - 1
matching_error = np.max(np.abs(raw_minmax - sample[0]))
np.testing.assert_allclose(raw_minmax, sample[0], rtol=1e-5, atol=1e-6)
print("Source-to-NPY maximum absolute difference:", matching_error)
print("Source record and stored SYMILE ECG match.")

## Apply the repository preprocessing
Load `filter_bandpass` directly from ECGFounder's code. It applies a 50 Hz notch, a fourth-order 0.67–40 Hz bandpass and median-filter baseline removal. Then apply **one z-score normalization over the complete 12-lead recording**, as in their dataset class. This is not per-lead normalization.

We compare preprocessing of the original waveform with preprocessing of SYMILE's min-max waveform. Because this is an adapted NPY input, the comparison checks its numerical agreement for this sample; it does not assume all preprocessing operations are perfectly interchangeable for every recording. The repository's undefined `sample_rate` resampling call is not used: this header already verifies 500 Hz and 5,000 samples.

In [ ]:
def load_repo_module(filename, module_name):
    # Unique names avoid collisions with other projects' generic util/net1d modules.
    spec = importlib.util.spec_from_file_location(module_name, ECGFOUNDER_DIR / filename)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

founder_util = load_repo_module("util.py", "ecgfounder_reference_util")
founder_net = load_repo_module("net1d.py", "ecgfounder_reference_net1d")


def preprocess_ecg(time_by_lead, source_leads=SOURCE_LEADS, sampling_rate=FS):
    # Recompute indices for each record rather than assuming one common order.
    indices = [source_leads.index(name) for name in TARGET_LEADS]
    channels_first = np.asarray(time_by_lead, dtype=np.float64).T[indices]
    filtered = founder_util.filter_bandpass(channels_first, sampling_rate)
    if not np.isfinite(filtered).all() or filtered.std() == 0:
        raise ValueError("The filtered ECG is nonfinite or constant.")
    normalized = (filtered - filtered.mean()) / (filtered.std() + 1e-8)
    return normalized.astype(np.float32)

processed_npy = preprocess_ecg(sample[0])
processed_raw = preprocess_ecg(raw_signal)
waveform_max_error = float(np.max(np.abs(processed_npy - processed_raw)))
waveform_rmse = float(np.sqrt(np.mean((processed_npy - processed_raw) ** 2)))
assert processed_npy.shape == (12, 5000)
assert np.isfinite(processed_npy).all()
print("Adapted NPY shape:", processed_npy.shape)
print("Mean / standard deviation:", processed_npy.mean(), processed_npy.std())
print("NPY versus raw preprocessing maximum absolute difference:", waveform_max_error)
print("NPY versus raw preprocessing RMSE:", waveform_rmse)
display(pd.DataFrame({
    "source": ["SYMILE min-max", "ECGFounder adapted NPY"],
    "minimum": [sample.min(), processed_npy.min()],
    "maximum": [sample.max(), processed_npy.max()],
    "mean": [sample.mean(), processed_npy.mean()],
    "std": [sample.std(), processed_npy.std()],
}))

## Inspect the preprocessed leads
The horizontal axis is seconds. Amplitudes are dimensionless after record-wide z-score normalization.

In [ ]:
seconds = np.arange(processed_npy.shape[1]) / FS
fig, axes = plt.subplots(6, 2, figsize=(14, 11), sharex=True)
for i, ax in enumerate(axes.flat):
    ax.plot(seconds, processed_npy[i], linewidth=0.7)
    ax.set_title(TARGET_LEADS[i])
    ax.grid(alpha=0.2)
for ax in axes[-1]:
    ax.set_xlabel("Time (seconds)")
fig.suptitle("One SYMILE ECG after ECGFounder preprocessing")
fig.tight_layout()
plt.show()
plt.close(fig)

## Load the pretrained weights
Use the exact 12-lead architecture from `ft_12lead_ECGFounder`, including `use_bn=False` and `use_do=False`. Keep the original 150-output head only so we can load **every checkpoint key strictly**. Set `return_features=True` to obtain `(logits, deep_features)`; our desired embedding is `deep_features`, before the classifier.

The checksum below is the official Hugging Face SHA256 for the downloaded release. The checkpoint is loaded in evaluation mode with gradients disabled. No optimizer is created.

In [ ]:
EXPECTED_SHA256 = "ee199f3781f4ae1f732973267f003da0a759ea12bddb0dd28a77faa60aca7997"
digest = hashlib.sha256()
with CHECKPOINT_PATH.open("rb") as stream:
    for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
        digest.update(chunk)
actual_sha256 = digest.hexdigest()
if actual_sha256 != EXPECTED_SHA256:
    raise ValueError("Checkpoint checksum differs from the verified official release.")
print("Official checkpoint checksum verified.")

# weights_only=False is used only for this checksum-verified trusted checkpoint.
checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
state_dict = checkpoint["state_dict"]
assert tuple(state_dict["dense.weight"].shape) == (150, 1024)
encoder = founder_net.Net1D(
    in_channels=12, base_filters=64, ratio=1,
    filter_list=[64, 160, 160, 400, 400, 1024, 1024],
    m_blocks_list=[2, 2, 2, 3, 3, 4, 4],
    kernel_size=16, stride=2, groups_width=16,
    n_classes=150, use_bn=False, use_do=False,
    return_features=True, verbose=False,
)
encoder.load_state_dict(state_dict, strict=True)
encoder.to(DEVICE).eval()
encoder.requires_grad_(False)
del checkpoint, state_dict
print("Loaded all pretrained weights strictly.")
print("Encoder parameters:", f"{sum(p.numel() for p in encoder.parameters()):,}")
print("Feature dimension:", encoder.dense.in_features)

## Extract the 1024 dimensional embedding
Convert `(12, 5000)` into the batch shape `(1, 12, 5000)`. Compare embeddings from the adapted NPY input and raw-reference preprocessing. The 150 original diagnosis logits are not our 14 CheXpert predictions. This notebook uses only the embedding and does not yet add shared/distinct branches.

In [ ]:
ecg_input = torch.from_numpy(processed_npy).unsqueeze(0).to(DEVICE)
raw_input = torch.from_numpy(processed_raw).unsqueeze(0).to(DEVICE)
with torch.inference_mode():
    original_logits, embedding = encoder(ecg_input)
    _, raw_embedding = encoder(raw_input)

assert ecg_input.shape == (1, 12, 5000)
assert embedding.shape == (1, 1024)
assert torch.isfinite(embedding).all()
cosine_agreement = torch.nn.functional.cosine_similarity(embedding, raw_embedding).item()
embedding_max_error = (embedding - raw_embedding).abs().max().item()
print("ECGFounder input:", tuple(ecg_input.shape))
print("Original diagnosis logits (unused):", tuple(original_logits.shape))
print("Pretrained embedding:", tuple(embedding.shape))
print("Embedding norm:", embedding.norm().item())
print("NPY / raw embedding cosine agreement:", cosine_agreement)
print("NPY / raw embedding maximum absolute difference:", embedding_max_error)
print("First 12 embedding values:", embedding[0, :12].cpu())

# A numerical compatibility check for this single sample, not an accuracy test.
np.testing.assert_allclose(processed_npy, processed_raw, atol=1e-3, rtol=1e-3)
assert cosine_agreement > 0.9999
print("Single-sample input, preprocessing and pretrained feature checks passed.")

## Check 20 additional training recordings
Select 20 distinct patients with seed 42, excluding the original sample's patient. The checks use no validation or test admissions. Download at most four independent records concurrently; existing cached records are reused.

Tolerances are declared **before testing**: raw-to-NPY agreement within `atol=1e-6, rtol=1e-5`; processed-waveform agreement within `atol=1e-3, rtol=1e-3`; embedding cosine similarity above `0.9999`; embedding relative L2 difference below `0.001`. These are numerical compatibility checks, not clinical equivalence or prediction-accuracy criteria. Download failures and numerical mismatches are recorded separately and are not silently replaced with easier samples.

In [ ]:
NUM_ADDITIONAL_SAMPLES = 20
CHECK_SEED = 42
RAW_ATOL, RAW_RTOL = 1e-6, 1e-5
PROCESSED_ATOL, PROCESSED_RTOL = 1e-3, 1e-3
MIN_EMBEDDING_COSINE = 0.9999
MAX_EMBEDDING_RELATIVE_L2 = 1e-3

candidates = metadata.loc[metadata["subject_id"].ne(row["subject_id"])]
candidates = candidates.drop_duplicates("subject_id")
selected = candidates.sample(n=NUM_ADDITIONAL_SAMPLES, random_state=CHECK_SEED)
selected_indices = selected.index.to_list()
assert SAMPLE_INDEX not in selected_indices
assert selected["subject_id"].is_unique
print("Additional training indices:", selected_indices)
print("Distinct additional patients:", selected["subject_id"].nunique())

In [ ]:
def download_selected_record(index):
    try:
        path = cached_source_record(metadata.iloc[index]["ecg_path"])
        return index, path, None
    except (requests.RequestException, OSError, ValueError) as error:
        return index, None, f"{type(error).__name__}: {error}"


def compare_record(index, path):
    stored = np.asarray(ecg_array[index, 0], dtype=np.float64)
    record = wfdb.rdrecord(str(path))
    raw = record.p_signal.astype(np.float64)
    if record.fs != 500 or record.n_sig != 12 or raw.shape != (5000, 12):
        raise ValueError("Unexpected source sampling rate, channel count or shape.")
    if len(set(record.sig_name)) != 12 or set(record.sig_name) != set(TARGET_LEADS):
        raise ValueError("Missing, repeated or unexpected ECG lead names.")
    if stored.shape != raw.shape or not np.isfinite(raw).all() or not np.isfinite(stored).all():
        raise ValueError("Stored/source ECG shape or finite-value check failed.")
    span = raw.max() - raw.min()
    if span <= 0:
        raise ValueError("Constant source ECG.")
    normalized_raw = 2 * (raw - raw.min()) / span - 1
    aligned = np.allclose(stored, normalized_raw, atol=RAW_ATOL, rtol=RAW_RTOL)
    if not aligned:
        # Wrong source alignment must not be presented as a preprocessing result.
        return {"status": "mismatch", "error": "Source and NPY waveforms do not align",
                "source_to_npy_max_abs": float(np.abs(stored - normalized_raw).max())}

    npy_processed = preprocess_ecg(stored, record.sig_name, record.fs)
    raw_processed = preprocess_ecg(raw, record.sig_name, record.fs)
    difference = npy_processed.astype(np.float64) - raw_processed
    waveform_agrees = np.allclose(npy_processed, raw_processed,
                                 atol=PROCESSED_ATOL, rtol=PROCESSED_RTOL)
    normalization_ok = abs(float(npy_processed.mean())) < 1e-5 and abs(float(npy_processed.std()) - 1) < 1e-3
    pair = torch.from_numpy(np.stack([npy_processed, raw_processed])).to(DEVICE)
    with torch.inference_mode():
        _, features = encoder(pair)
    if features.shape != (2, 1024) or not torch.isfinite(features).all():
        raise ValueError("Encoder output has an incorrect shape or nonfinite values.")
    # Calculate similarities in float64 to expose differences hidden by rounding.
    adapted, reference = features.cpu().double()
    reference_norm = reference.norm().item()
    if reference_norm == 0 or adapted.norm().item() == 0:
        raise ValueError("Zero-norm embedding cannot establish feature agreement.")
    cosine = float(torch.nn.functional.cosine_similarity(adapted[None], reference[None]))
    relative_l2 = (adapted - reference).norm().item() / reference_norm
    passed = waveform_agrees and normalization_ok and cosine > MIN_EMBEDDING_COSINE and relative_l2 < MAX_EMBEDDING_RELATIVE_L2
    return {
        "status": "pass" if passed else "mismatch",
        "fs_hz": record.fs,
        "source_lead_order": ",".join(record.sig_name),
        "source_to_npy_max_abs": float(np.abs(stored - normalized_raw).max()),
        "processed_max_abs": float(np.abs(difference).max()),
        "processed_RMSE": float(np.sqrt(np.mean(difference ** 2))),
        "embedding_cosine": cosine,
        "embedding_max_abs": (adapted - reference).abs().max().item(),
        "embedding_relative_L2": relative_l2,
        "waveform_agrees": waveform_agrees,
        "normalization_ok": normalization_ok,
        "error": "" if passed else "At least one declared numerical check failed",
    }

check_rows = []
with ThreadPoolExecutor(max_workers=4) as pool:
    for count, (index, path, download_error) in enumerate(pool.map(download_selected_record, selected_indices), 1):
        info = {"array_index": index,
                "subject_id": int(metadata.iloc[index]["subject_id"]),
                "hadm_id": int(metadata.iloc[index]["hadm_id"]),
                "ecg_path": str(metadata.iloc[index]["ecg_path"])}
        if download_error:
            result = {"status": "download_error", "error": download_error}
        else:
            try:
                result = compare_record(index, path)
            except (ValueError, AssertionError, RuntimeError, OSError) as error:
                result = {"status": "check_error", "error": f"{type(error).__name__}: {error}"}
        check_rows.append({**info, **result})
        print(f"{count}/{NUM_ADDITIONAL_SAMPLES}: row {index} — {result['status']}")

validation_results = pd.DataFrame(check_rows)
report_path = DATASET_ROOT / "ecgfounder_sample_checks/preprocessing_validation_20.csv"
validation_results.to_csv(report_path, index=False)
print("Detailed report:", report_path)

## Additional sample results
The table reports every selected recording, including errors. Similarity alone does not test embedding magnitude, so relative L2 difference is reported alongside cosine similarity. The summary includes both attempted and successful comparisons.

In [ ]:
columns = ["array_index", "hadm_id", "status", "source_to_npy_max_abs",
           "processed_max_abs", "processed_RMSE", "embedding_cosine",
           "embedding_max_abs", "embedding_relative_L2", "error"]
with pd.option_context("display.max_rows", 25, "display.max_columns", None, "display.float_format", "{:.9g}".format):
    display(validation_results.reindex(columns=columns))

completed = validation_results.loc[validation_results["status"].isin(["pass", "mismatch"])]
metrics = ["source_to_npy_max_abs", "processed_max_abs", "processed_RMSE",
           "embedding_cosine", "embedding_max_abs", "embedding_relative_L2"]
numerical_summary = validation_results.reindex(columns=metrics).agg(["min", "mean", "max"]).T
print("Selected:", len(validation_results))
print("Passed:", int(validation_results["status"].eq("pass").sum()))
print("Numerical mismatches:", int(validation_results["status"].eq("mismatch").sum()))
print("Download / check errors:", int(validation_results["status"].isin(["download_error", "check_error"]).sum()))
with pd.option_context("display.float_format", "{:.9g}".format):
    display(numerical_summary)
if validation_results["status"].eq("pass").all():
    print("All 20 additional ECGs passed the declared numerical compatibility checks.")
else:
    print("Inspect the nonpassing rows before choosing the preprocessing pipeline.")

## Scope of the result
These checks cover the original demonstration ECG and 20 additional training patients. They verify waveform alignment, source metadata, lead reordering, the repository preprocessing and the numerical agreement of pretrained embeddings. Results quantify agreement for the sampled recordings only; they do not prove equivalence over the full dataset or an improvement in AP/AUROC.

The next model-design step is to add learned shared and distinct projections from 1,024 to 256 dimensions, plus our 14-finding auxiliary head. No fine-tuning, model integration or changes to the original arrays or patient splits are performed by this notebook.